# Phase 3: Lexical Baseline Retrieval (BM25 & TF-IDF)

**Project**: Enhanced Legal Document Similarity and Case Retrieval Using Hybrid BERT, Legal Knowledge, and Ensemble Learning  
**Dataset**: `data/processed/IL-TUR-PCR/` (Prior Case Retrieval)  
**Objective**: Implement, evaluate, and compare BM25 and TF-IDF baselines with chunk-to-document aggregation and query self-exclusion.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Setup project root
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.tfidf_retriever import TFIDFRetriever
from src.retrieval.document_retriever import DocumentRetriever
from src.evaluation.retrieval_metrics import (
    evaluate_query_ranking,
    aggregate_retrieval_metrics,
)
print("Modules imported successfully.")

## 1. Load Processed IL-TUR PCR Partition Data
We load `dev_queries` and `dev_candidates` (both `documents.parquet` and `chunks.parquet`).

In [ ]:
data_dir = PROJECT_ROOT / "data" / "processed" / "IL-TUR-PCR"

dev_queries_df = pd.read_parquet(data_dir / "dev_queries" / "documents.parquet")
dev_cand_docs_df = pd.read_parquet(data_dir / "dev_candidates" / "documents.parquet")
dev_cand_chunks_df = pd.read_parquet(data_dir / "dev_candidates" / "chunks.parquet")

print(f"Dev Queries: {len(dev_queries_df)} cases")
print(f"Dev Candidate Documents: {len(dev_cand_docs_df)} cases")
print(f"Dev Candidate Chunks: {len(dev_cand_chunks_df)} chunks (avg {len(dev_cand_chunks_df)/len(dev_cand_docs_df):.1f} chunks/doc)")

## 2. Fit BM25 and TF-IDF on Candidate Chunks
Both retrievers use sparse CSR matrices for memory-efficient indexing and vectorized dot-product scoring.

In [ ]:
# Fit BM25
bm25 = BM25Retriever(k1=1.5, b=0.75, max_features=50000, min_df=2, max_df=0.9)
bm25.fit(dev_cand_chunks_df["text"].values)
print("BM25 Index Stats:", bm25.get_index_stats())

# Fit TF-IDF
tfidf = TFIDFRetriever(ngram_range=(1, 1), min_df=2, max_df=0.9, max_features=50000)
tfidf.fit(dev_cand_chunks_df["text"].values)
print("TF-IDF Index Stats:", tfidf.get_index_stats())

## 3. Score Queries against Chunks
We compute query-chunk similarity matrices for all 118 dev queries.

In [ ]:
query_texts = dev_queries_df["full_text"].values

bm25_chunk_scores = bm25.score_queries(query_texts, batch_size=256).toarray()
tfidf_chunk_scores = tfidf.score_queries(query_texts, batch_size=256).toarray()

print("BM25 Chunk Scores shape:", bm25_chunk_scores.shape)
print("TF-IDF Chunk Scores shape:", tfidf_chunk_scores.shape)

## 4. Chunk-to-Document Aggregation & Query Self-Exclusion
We initialize `DocumentRetriever` to map candidate chunks to candidate documents and enforce query self-exclusion.

In [ ]:
doc_retriever = DocumentRetriever(
    candidate_doc_ids=dev_cand_docs_df["id"].values,
    chunk_doc_ids=dev_cand_chunks_df["doc_id"].values,
)

# Inspect first dev query
sample_q = dev_queries_df.iloc[0]
qid = sample_q["id"]
ground_truth = set(sample_q["relevant_candidates"])
print(f"Sample Query ID: {qid}")
print(f"Ground Truth Relevant Precedents ({len(ground_truth)}): {list(ground_truth)[:5]}...")

# Aggregation comparisons
max_scores = doc_retriever.aggregate_chunk_scores(bm25_chunk_scores[0], strategy="max")
top3_scores = doc_retriever.aggregate_chunk_scores(bm25_chunk_scores[0], strategy="top_k_mean", top_k=3)

# Rank with query self-exclusion
ranked_max = doc_retriever.rank_candidates(max_scores, query_id=qid, top_n=5)
ranked_top3 = doc_retriever.rank_candidates(top3_scores, query_id=qid, top_n=5)

print("\n--- Top 5 Retrieved (BM25 Chunk-Max) ---")
for rank, (cand_id, score) in enumerate(ranked_max, start=1):
    is_hit = "[RELEVANT HIT]" if cand_id in ground_truth else ""
    print(f"{rank}. Doc ID: {cand_id} | Score: {score:.2f} {is_hit}")

print("\n--- Top 5 Retrieved (BM25 Chunk-Top3Mean) ---")
for rank, (cand_id, score) in enumerate(ranked_top3, start=1):
    is_hit = "[RELEVANT HIT]" if cand_id in ground_truth else ""
    print(f"{rank}. Doc ID: {cand_id} | Score: {score:.2f} {is_hit}")

## 5. Consolidated Benchmark Results across Splits
Let's load and display the full baseline comparison table generated by `BaselineEvaluator`.

In [ ]:
comp_csv = PROJECT_ROOT / "results" / "tables" / "baseline_comparison.csv"
df_results = pd.read_csv(comp_csv)
display(df_results.head(16))

## 6. Chunk Aggregation Comparison
Inspect how `Chunk-Max`, `Top3Mean`, and `Top5Mean` compare across Dev and Test splits.

In [ ]:
agg_csv = PROJECT_ROOT / "results" / "tables" / "chunk_aggregation_comparison.csv"
df_agg = pd.read_csv(agg_csv)
display(df_agg)